# RAN Split

## Prerequisites

Before starting, make sure you:

- Have completed the [Simple Scenario](./10_simple.ipynb) and [Adding component](./11_add_components.ipynb) tutorials.
- Have a basic understanding of Python.
- Have a good understanding of the 5G architecture.
- Have a good understanding of the RAN split

## Experiment for this scenario

In this scenario we implement RAN splitting. We will consider the case of the
*CU - DU* split and the more advance case of *CU-CP/CU-UP - DU* split.

In our network on the one hand, the DU named `du1` is connected to the CU named
`cu`. On the other hand, the DU named `du2` is connected to the CU-CP named 
`cucp` and to the CU-UP named `cuup`. The UE `sim_ue1` is connected to `du1` and
the UE `sim_ue2` is connected to `du2` as illustrated in the figure below.

Each DU has its own Cell ID and PCI.

The 5G core is implemented with `Open5GS`, and the RAN is implemented with OAI.
The two UEs are implemented with `OAI` and the  radio channel is simulated with
rfsim.

![Split topology](figures/split.png)





## Metamodel and Deployement model definition

### Metamodel

The metamodel follows the same principle as in the other scenarios, except that
instead of defining a monolithic gNB, we define the split functions and how they
are connected.

We declare that the DU `du1` is connected to the CU `cu` by adding the `cu_cp`
and `cu_up` attributes to its `config` section. As `cu` implements at the same
time the CP and the UP, we reference it in both case.

On the contrary the DU `du2` is connected to the CU-CP `cucp` and to the CU-UP
`cuup`, we then explicitly reference them in the the `cu_cp` and `cu_up`
attributes of the DU. In addition, we have to specify that `cuup` has an E1
interface with the CU-CP `cucp`. This is why we have to add the `cu_cp`
attribute to the `config` section of `cuup`.

In addition, we specify a specific DU ID, CellID, and PCI to each DU for the
network to distinguish them.

In [1]:
from ifran import MetaModel

metamodel = MetaModel.from_yaml(
"""
ran:
    components:
      cu:
        type: cu

      cucp:
        type: cu_cp

      cuup:
        type: cu_up
        config:
          cu_cp: cucp

      du1:
        type: du
        config:
          gnbDuId: 1
          nrCellId: 1
          physicalCellId: 1
          cu_cp: cu
          cu_up: cu

      du2:
        type: du
        config:
          gnbDuId: 2
          nrCellId: 2
          physicalCellId: 2
          cu_cp: cucp
          cu_up: cuup
    
ues:
  sim_ue1:
    config:
      supi: '001010000000100'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - du1
  sim_ue2:
    config:
      supi: '001010000000101'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - du2
""")

### Deployment model

Beside declaring each component constituting the split RAN, there is no
specific settings to add to the deployment model to implement the split.

In [2]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
ran:
  implementation: oai
  components:
    cu:
      type: cu
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
    du1:
      type: du
    du2:
      type: du

ues:
  sim_ue1:
    implementation: oai
    type: containerized
  sim_ue2:
    implementation: oai
    type: containerized

nodes:
  node-core:
    provisioner: my-provisionner
    roles:
    - core
    - ric
  node-ran:
    provisioner: my-provisionner
    roles:
    - ran
  node-ues:
    provisioner: my-provisionner
    roles:
    - ue

provisioners:
  my-provisionner:
    type: bi
    params:
      infra: be-gent1-bi-vm1
      project: post5g-beta
      experiment: split_scenario
      create_experiment: true
      experiment_duration: 1h
      node_duration: 1h
"""
)



## Provisioning the infrastructure

We can provision the infrastructure as in the [Simple Scenario tutorial](./10_simple.ipynb).

In [3]:
from ifran import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-core' provisioning] ***

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' f

At this stage, the infrastructure is fully ready to support our 5G network
deployment.

But before deploying the 5G network, let's archive the metamodel and deployment
model resulting from this deployment.

In [4]:
from pathlib import Path
DIR="state/split"
Path(DIR).mkdir(parents=True, exist_ok=True)

In [5]:
prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

## Deploy the 5G Network

### Deploy the 5G core

Now that the infrastructure is fully provisioned, we first have to deploy the 5G
core.

In [6]:
from ifran import CoreManagement

core = CoreManagement(preceding=k8s)
await core.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nssf' chart] *****

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-upf' chart] ******

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nrf' chart] ******

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-smf' chart] ******

PLAY [node-co

### Deploy the 5G RAN

We can now deploy the 5G RAN and verify that the RAN split functions are
deployed properly.

In [7]:
prov = ProvisioningManagement(preceding=core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core", "-o", "wide"],
    nodes=["node-core"],
    become=True, quiet=True)))

NAME                            READY   STATUS    RESTARTS   AGE     IP               NODE        NOMINATED NODE   READINESS GATES
open5gs-amf-7b6dcfbfdf-j8xrc    1/1     Running   0          116s    10.244.134.7     node-core   <none>           <none>
open5gs-ausf-5778d7548b-ztwcx   1/1     Running   0          101s    10.244.134.8     node-core   <none>           <none>
open5gs-bsf-75cf9d7fb4-txxfd    1/1     Running   0          15s     10.244.105.131   node-core   <none>           <none>
open5gs-database-0              1/1     Running   0          2m36s   10.244.134.5     node-core   <none>           <none>
open5gs-nrf-65d98494d7-69rgc    1/1     Running   0          2m14s   10.244.134.6     node-core   <none>           <none>
open5gs-nssf-84cb8d64f7-lr9cf   1/1     Running   0          30s     10.244.105.130   node-core   <none>           <none>
open5gs-pcf-6f444b66c5-42hs5    1/1     Running   0          74s     10.244.134.10    node-core   <none>           <none>
open5gs-smf-5c7

In [8]:
from ifran import RANManagement

ran = RANManagement(preceding=core)
await ran.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cu' chart] ***********

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du1' chart] **********

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du2' chart] **********
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different inter

We can verify that all functions are deployed in the kubernetes cluster by
checking the pods deployed in the `oai-ran` namespace.

In [9]:
print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=[prov.deployment_model.params.k8s.master_node],
    become=True, quiet=True, flatten=False))[prov.deployment_model.params.k8s.master_node])

NAME                       READY   STATUS    RESTARTS   AGE   IP               NODE       NOMINATED NODE   READINESS GATES
oai-cu-55bc5698-fb29z      1/1     Running   0          22s   10.244.91.223    node-ran   <none>           <none>
oai-cucp-d5cfbf6dd-kvrsz   1/1     Running   0          22s   10.244.104.156   node-ran   <none>           <none>
oai-cuup-b8bf46577-j5zl5   1/1     Running   0          10s   10.244.74.21     node-ran   <none>           <none>
oai-du1-74bbfcb8d7-79ccs   1/1     Running   0          3s    10.244.65.96     node-ran   <none>           <none>
oai-du2-769bbb95fc-lcfnt   1/1     Running   0          3s    10.244.253.60    node-ran   <none>           <none>


## Deploy the UEs

Now that we have a fully functional core and RAN, we can deploy the UEs we have
defined in our metamodel.

In [10]:
from ifran import UEManagement

ues = UEManagement(preceding=ran)
await ues.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue2' chart] ******

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue1' chart] ******
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]
changed

We can check that the two UEs have been deployed by listing the pods in the
`oai-ue` namespace.

In [11]:
print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ue", "-o", "wide"],
    nodes=["node-ues"],
    become=True, quiet=True)))

NAME                           READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
oai-sim-ue1-555b9bc854-ksrrc   1/1     Running   0          12s   10.244.198.3   node-ues   <none>           <none>
oai-sim-ue2-c445dc77c-fh6kr    1/1     Running   0          12s   10.244.198.2   node-ues   <none>           <none>


We can also analyse the logs the the UEs and see that they both successuflly
registered to the network and established a PDU session.

In [12]:
print(await ues.get_logs(components=["sim_ue1"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-f80a88326801f5417405dea0b174879bac8940e0 Ab

In [13]:
print(await ues.get_logs(components=["sim_ue2"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-f80a88326801f5417405dea0b174879bac8940e0 Ab

To test that the UEs are well connected and that our network works as expected
we can run a ping from the UEs via their radio interace. In UERANSIM, the
interface is called `uesimtun0`.

To run commands directly on the nodes,

In [14]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True, flatten=False)

ping from `sim_ue1`:

In [15]:
print (pings["sim_ue1"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 oaitun_ue1: 56(84) bytes of data.
[1794907777.746329] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=117 time=41.9 ms
[1794907778.745909] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=117 time=40.6 ms
[1794907779.760473] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=117 time=53.5 ms
[1794907780.741801] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=117 time=33.1 ms
[1794907781.748095] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=117 time=38.4 ms
[1794907782.752692] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=117 time=42.3 ms
[1794907783.761331] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=117 time=50.4 ms
[1794907784.738966] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=117 time=27.1 ms
[1794907785.746130] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=117 time=33.8 ms
[1794907786.749185] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=117 time=35.8 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9009ms
rtt min/avg/max/mdev = 27.087/39.675/53.510/7.560 ms



ping from `sim_ue2`:

In [16]:
print (pings["sim_ue2"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 oaitun_ue1: 56(84) bytes of data.
[1794907777.794457] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=117 time=38.8 ms
[1794907778.786395] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=117 time=30.8 ms
[1794907779.800631] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=117 time=44.3 ms
[1794907780.781539] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=117 time=25.1 ms
[1794907781.796430] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=117 time=39.8 ms
[1794907782.794341] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=117 time=37.0 ms
[1794907783.807024] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=117 time=49.7 ms
[1794907784.795793] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=117 time=38.4 ms
[1794907785.792923] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=117 time=35.1 ms
[1794907786.793921] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=117 time=36.1 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9003ms
rtt min/avg/max/mdev = 25.092/37.500/49.664/6.387 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [17]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")